# 04.1 — Segundo intento: mejora del clasificador grueso

Este cuaderno preserva el baseline documentado en `resultados/INFORME_PRIMER_ENTRENAMIENTO_MODELOS_GRUESOS.md` y evalúa mejoras sin entrenar etiquetas finas ni flags. El único objetivo sigue siendo **cinco daños gruesos o SEGURO**.

Mejoras aplicadas: contexto del título y chunks vecinos; ajuste de regularización; menor peso de los seguros pseudoetiquetados por Flash; una ablation de aumentación AEDA sobre daños del entrenamiento; umbrales orientados a recall; y minería de seguros Flash difíciles para revisión futura.

In [ ]:
%pip install -q "pandas>=2.2,<3" "numpy>=1.26,<3" "scipy>=1.13,<2" "scikit-learn>=1.5,<2" "joblib>=1.4,<2" "matplotlib>=3.9,<4"

## 1. Configuración y rutas

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
for candidate in (cwd, *cwd.parents):
    if (candidate / 'scripts_auxiliares' / 'mejoras_modelos_gruesos.py').exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('No se encontró la raíz del proyecto.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts_auxiliares.flujo_hibrido_moderador import (
    build_hybrid_dataset, grouped_train_validation_test_split, load_taxonomy,
)
from scripts_auxiliares.modelos_gruesos_moderador import (
    COARSE_ORDER, DAMAGE_ORDER, add_coarse_targets, evaluate_candidate,
    fit_candidate, save_coarse_model, target_matrix, tune_candidate,
)
from scripts_auxiliares.mejoras_modelos_gruesos import (
    add_neighbor_context, augment_damage_with_punctuation,
    evaluate_threshold_policy, mine_flash_hard_negatives,
    routing_with_damage_and_uncertainty, save_hard_negative_manifest,
    tune_thresholds_for_minimum_recall,
)

SEED = 42
BASELINE_SPLIT_SEED = 131
MAX_FEATURES = 50_000
MINIMUM_RECALL = 0.80
OUTPUT_METRICS = ROOT / 'resultados' / 'metricas' / 'moderador_grueso_mejorado'
OUTPUT_FIGURES = ROOT / 'resultados' / 'figuras' / 'moderador_grueso_mejorado'
OUTPUT_MODEL = ROOT / 'modelos' / 'moderador_grueso_mejorado'
for directory in (OUTPUT_METRICS, OUTPUT_FIGURES, OUTPUT_MODEL):
    directory.mkdir(parents=True, exist_ok=True)
np.random.seed(SEED)
print('Raíz:', ROOT)

## 2. Reconstrucción exacta del corpus adjudicado y de la partición baseline

Se aplica la precedencia humano grueso > Pro > Flash. El flujo se bloquea hasta que las 139 dudas persistentes de Pro tengan decisión humana completa; 114 pertenecen a train, 25 a validation y ninguna a test. Las etiquetas finas se conservan solo como referencia y nunca se entrenan; los flags transversales se mantienen separados de las categorías base. Se conserva el mismo test del primer intento para medir el cambio directamente. La selección de configuraciones usa validación; el test solo se evalúa después de fijar el ganador. Como el test ya fue observado en el informe baseline, este segundo análisis es comparativo y no reemplaza un futuro holdout humano ciego.

In [ ]:
taxonomy, FINE_ORDER, FLAG_ORDER = load_taxonomy(ROOT)
hybrid_df, hybrid_meta, _, _ = build_hybrid_dataset(ROOT, write_output=False)
hybrid_df = add_coarse_targets(hybrid_df, taxonomy)
human_ids = set(hybrid_df.loc[hybrid_df['human_holdout'], 'chunk_id'])
modeling_pool = hybrid_df.loc[~hybrid_df['human_holdout']].reset_index(drop=True)
modeling_pool = add_neighbor_context(
    modeling_pool, ROOT / 'datos' / 'processed' / 'chunks_para_etiquetar.jsonl',
    radius=1, include_title=True, output_column='context_text',
)
split = grouped_train_validation_test_split(
    modeling_pool, seed=BASELINE_SPLIT_SEED, test_size=0.15, validation_size=0.15
)
train_df = modeling_pool.iloc[split['train']].reset_index(drop=True)
validation_df = modeling_pool.iloc[split['validation']].reset_index(drop=True)
test_df = modeling_pool.iloc[split['test']].reset_index(drop=True)
assert hybrid_meta['human_hard_rows'] == 139 and hybrid_meta['hard_review_complete']
assert (modeling_pool['label_source'] == 'human_coarse').sum() == 139
assert (train_df['label_source'] == 'human_coarse').sum() == 114
assert (validation_df['label_source'] == 'human_coarse').sum() == 25
assert (test_df['label_source'] == 'human_coarse').sum() == 0
assert (len(train_df), len(validation_df), len(test_df)) == (48_927, 10_633, 10_293)
assert set(train_df.video_id).isdisjoint(set(validation_df.video_id))
assert set(train_df.video_id).isdisjoint(set(test_df.video_id))
assert set(validation_df.video_id).isdisjoint(set(test_df.video_id))
print('Partición reproducida:', len(train_df), len(validation_df), len(test_df))
print('Longitud mediana texto actual:', int(train_df.text.str.len().median()))
print('Longitud mediana con contexto:', int(train_df.context_text.str.len().median()))

## 3. Búsqueda controlada de configuraciones

Se comparan SVM y regresión logística, los dos mejores algoritmos del baseline. La búsqueda varía representación (`text` o `context_text`), regularización `C` y peso base de Flash. No se usan etiquetas finas ni flags como entrada. La selección primaria usa PR-AUC macro de daño, que no depende de un único umbral y es informativa bajo desbalance (Saito & Rehmsmeier, 2015).

In [ ]:
CONFIGURATIONS = [
    {'id': 'svm_text_c1_w50_baseline', 'model': 'linear_svm_word_char', 'text_column': 'text', 'C': 1.0, 'flash_weight': 0.50},
    {'id': 'svm_context_c03_w50', 'model': 'linear_svm_word_char', 'text_column': 'context_text', 'C': 0.3, 'flash_weight': 0.50},
    {'id': 'svm_context_c1_w50', 'model': 'linear_svm_word_char', 'text_column': 'context_text', 'C': 1.0, 'flash_weight': 0.50},
    {'id': 'svm_context_c03_w25', 'model': 'linear_svm_word_char', 'text_column': 'context_text', 'C': 0.3, 'flash_weight': 0.25},
    {'id': 'svm_context_c1_w25', 'model': 'linear_svm_word_char', 'text_column': 'context_text', 'C': 1.0, 'flash_weight': 0.25},
    {'id': 'logreg_context_c03_w25', 'model': 'logistic_regression', 'text_column': 'context_text', 'C': 0.3, 'flash_weight': 0.25},
    {'id': 'logreg_context_c1_w25', 'model': 'logistic_regression', 'text_column': 'context_text', 'C': 1.0, 'flash_weight': 0.25},
    {'id': 'logreg_context_c1_w50', 'model': 'logistic_regression', 'text_column': 'context_text', 'C': 1.0, 'flash_weight': 0.50},
]

models = {}
validation_scores = {}
validation_reports = {}
rows = []
for spec in CONFIGURATIONS:
    print('Entrenando', spec['id'])
    model, diagnostics = fit_candidate(
        spec['model'], train_df, max_features=MAX_FEATURES,
        text_column=spec['text_column'], model_parameters={'C': spec['C']},
        flash_pseudo_weight=spec['flash_weight'],
    )
    tune_candidate(model, validation_df, text_column=spec['text_column'])
    metrics, report, scores = evaluate_candidate(
        model, validation_df, text_column=spec['text_column']
    )
    models[spec['id']] = model
    validation_scores[spec['id']] = scores
    validation_reports[spec['id']] = report
    rows.append({**spec, **diagnostics, **{f'val_{key}': value for key, value in metrics.items()}})

search_results = pd.DataFrame(rows).sort_values(
    ['val_damage_pr_auc_macro', 'val_damage_f1_macro'], ascending=False
).reset_index(drop=True)
display(search_results[[
    'id', 'training_seconds', 'val_damage_pr_auc_macro',
    'val_damage_f1_macro', 'val_damage_recall_micro',
]])
search_results.to_csv(OUTPUT_METRICS / 'busqueda_configuraciones_sin_aumentacion.csv', index=False)

## 4. Ablation de aumentación sobre el mejor candidato

AEDA inserta puntuación sin reemplazar palabras. Se genera una copia de cada ejemplo de daño únicamente en entrenamiento, con peso 0.50; validación y prueba permanecen intactas. Esta técnica se evalúa, no se presupone beneficiosa (Karimi et al., 2021).

In [ ]:
best_without_augmentation = search_results.iloc[0].to_dict()
best_spec = next(spec for spec in CONFIGURATIONS if spec['id'] == best_without_augmentation['id'])
augmented_train = augment_damage_with_punctuation(
    train_df, text_column=best_spec['text_column'], seed=SEED,
    repetitions=1, insertion_rate=0.08, augmented_weight=0.50,
)
AUGMENTED_ID = best_spec['id'] + '_aeda'
augmented_model, augmented_diagnostics = fit_candidate(
    best_spec['model'], augmented_train, max_features=MAX_FEATURES,
    text_column=best_spec['text_column'], model_parameters={'C': best_spec['C']},
    flash_pseudo_weight=best_spec['flash_weight'],
)
tune_candidate(augmented_model, validation_df, text_column=best_spec['text_column'])
aug_metrics, aug_report, aug_scores = evaluate_candidate(
    augmented_model, validation_df, text_column=best_spec['text_column']
)
models[AUGMENTED_ID] = augmented_model
validation_scores[AUGMENTED_ID] = aug_scores
validation_reports[AUGMENTED_ID] = aug_report
aug_row = {
    **best_spec, 'id': AUGMENTED_ID, 'augmented': True,
    **augmented_diagnostics, **{f'val_{key}': value for key, value in aug_metrics.items()},
}
search_results = pd.concat([search_results, pd.DataFrame([aug_row])], ignore_index=True)
search_results = search_results.sort_values(
    ['val_damage_pr_auc_macro', 'val_damage_f1_macro'], ascending=False
).reset_index(drop=True)
WINNER_ID = search_results.iloc[0]['id']
winner_spec = aug_row if WINNER_ID == AUGMENTED_ID else next(spec for spec in CONFIGURATIONS if spec['id'] == WINNER_ID)
winner_model = models[WINNER_ID]
print('Mejor configuración por validación:', WINNER_ID)
print('Filas originales/aumentadas:', len(train_df), len(augmented_train))
display(search_results[['id', 'val_damage_pr_auc_macro', 'val_damage_f1_macro']])
search_results.to_csv(OUTPUT_METRICS / 'busqueda_configuraciones_completa.csv', index=False)

## 5. Evaluación comparativa en el test congelado

In [ ]:
test_rows = []
test_reports = {}
test_scores = {}
spec_by_id = {spec['id']: spec for spec in CONFIGURATIONS}
spec_by_id[AUGMENTED_ID] = winner_spec if WINNER_ID == AUGMENTED_ID else {**best_spec, 'id': AUGMENTED_ID}
for experiment_id, model in models.items():
    spec = spec_by_id[experiment_id]
    metrics, report, scores = evaluate_candidate(
        model, test_df, text_column=spec['text_column']
    )
    test_reports[experiment_id] = report
    test_scores[experiment_id] = scores
    test_rows.append({'id': experiment_id, **{f'test_{key}': value for key, value in metrics.items()}})
comparison = search_results.merge(pd.DataFrame(test_rows), on='id')
comparison = comparison.sort_values(
    ['val_damage_pr_auc_macro', 'val_damage_f1_macro'], ascending=False
)
display(comparison[[
    'id', 'val_damage_pr_auc_macro', 'test_damage_pr_auc_macro',
    'test_damage_f1_macro', 'test_damage_recall_micro', 'test_exact_match',
]])
comparison.to_csv(OUTPUT_METRICS / 'comparacion_mejoras_test.csv', index=False)
winner_test_report = test_reports[WINNER_ID]
display(winner_test_report.loc[COARSE_ORDER, ['precision', 'recall', 'f1-score', 'support']])

In [ ]:
baseline_f1 = 0.2751560959579362
baseline_pr_auc = 0.2317
plot_frame = comparison.head(9).copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
x = np.arange(len(plot_frame))
axes[0].bar(x - 0.18, plot_frame['test_damage_f1_macro'], 0.36, label='F1 macro daño')
axes[0].bar(x + 0.18, plot_frame['test_damage_pr_auc_macro'], 0.36, label='PR-AUC macro daño')
axes[0].axhline(baseline_f1, color='black', linestyle='--', label='F1 baseline')
axes[0].set_ylim(0, max(0.5, plot_frame['test_damage_f1_macro'].max() + 0.08))
axes[0].set_xticks(x, plot_frame['id'], rotation=35, ha='right')
axes[0].set_title('Configuraciones mejoradas en prueba')
axes[0].legend()

winner_matrix = winner_test_report.loc[COARSE_ORDER, ['precision', 'recall', 'f1-score']].to_numpy()
image = axes[1].imshow(winner_matrix, aspect='auto', vmin=0, vmax=1, cmap='YlOrRd')
axes[1].set_yticks(np.arange(len(COARSE_ORDER)), [name.replace('_', ' ') for name in COARSE_ORDER])
axes[1].set_xticks(np.arange(3), ['Precisión', 'Recall', 'F1'])
axes[1].set_title(f'Desempeño por salida: {WINNER_ID}')
for row in range(winner_matrix.shape[0]):
    for col in range(winner_matrix.shape[1]):
        axes[1].text(col, row, f'{winner_matrix[row, col]:.2f}', ha='center', va='center')
fig.colorbar(image, ax=axes[1], fraction=0.046)
fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / 'comparacion_mejoras.png', dpi=170, bbox_inches='tight')
plt.show()

![Comparación de mejoras](../resultados/figuras/moderador_grueso_mejorado/comparacion_mejoras.png)

## 6. Política de alta sensibilidad

La política baseline maximiza F1. Para priorización de moderación se construye otra política que, en validación, maximiza precisión sujeta a recall ≥ 0.80 por cada daño. Se informa la degradación de precisión y el volumen de revisión; no se confunde esta política con una mejora del ranking subyacente.

In [ ]:
high_recall_thresholds, threshold_table = tune_thresholds_for_minimum_recall(
    target_matrix(validation_df), validation_scores[WINNER_ID],
    minimum_recall=MINIMUM_RECALL,
)
high_recall_metrics, high_recall_report, high_recall_predictions = evaluate_threshold_policy(
    test_df, test_scores[WINNER_ID], high_recall_thresholds
)
display(threshold_table)
display(pd.Series(high_recall_metrics).to_frame('prueba_alta_sensibilidad'))
display(high_recall_report.loc[DAMAGE_ORDER, ['precision', 'recall', 'f1-score', 'support']])
threshold_table.to_csv(OUTPUT_METRICS / 'umbrales_alta_sensibilidad.csv', index=False)
high_recall_report.to_csv(OUTPUT_METRICS / 'reporte_alta_sensibilidad_test.csv')

In [ ]:
routing_rows = []
flag_any = test_df['flags'].map(bool).to_numpy()
true_damage = target_matrix(test_df)[:, 1:].any(axis=1)
for margin in np.linspace(0.01, 0.20, 20):
    review = routing_with_damage_and_uncertainty(
        test_scores[WINNER_ID], high_recall_thresholds, float(margin)
    )
    routing_rows.append({
        'margen': float(margin), 'tasa_revision': float(review.mean()),
        'captura_dano_real': float(review[true_damage].mean()),
        'captura_flags': float(review[flag_any].mean()),
        'precision_revision_dano': float(true_damage[review].mean()),
    })
routing = pd.DataFrame(routing_rows)
routing_choice = routing.loc[
    (routing['captura_dano_real'] >= 0.90) & (routing['captura_flags'] >= 0.80)
].sort_values('tasa_revision').head(1)
if routing_choice.empty:
    routing_choice = routing.sort_values(['captura_dano_real', 'captura_flags'], ascending=False).head(1)
display(routing_choice)
routing.to_csv(OUTPUT_METRICS / 'curva_revision_alta_sensibilidad.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(routing.tasa_revision, routing.captura_dano_real, label='Daño capturado')
ax.plot(routing.tasa_revision, routing.captura_flags, label='Flags capturados')
ax.scatter(routing_choice.tasa_revision, routing_choice.captura_dano_real, color='red', s=80)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Proporción enviada a revisión', ylabel='Captura', title='Costo de la política de alta sensibilidad')
ax.legend(); fig.tight_layout()
fig.savefig(OUTPUT_FIGURES / 'revision_alta_sensibilidad.png', dpi=170, bbox_inches='tight')
plt.show()

![Costo de la política de alta sensibilidad](../resultados/figuras/moderador_grueso_mejorado/revision_alta_sensibilidad.png)

## 7. Reajuste, exportación y minería de casos difíciles

El ganador se reajusta con entrenamiento + validación y se exporta con los umbrales de alta sensibilidad. La minería examina únicamente los seguros Flash de entrenamiento + validación; el test permanece excluido. Tras incorporar las 139 adjudicaciones, la nueva lista se escribe como iteración 2 y no sobrescribe los 2.000 casos que originaron la campaña humana. El manifiesto no cambia etiquetas ni llama a Pro: propone hasta 2.000 casos, máximo tres por video, para una eventual revisión independiente posterior.

In [ ]:
train_validation = pd.concat([train_df, validation_df], ignore_index=True)
winner_augmented = WINNER_ID.endswith('_aeda')
final_training = (
    augment_damage_with_punctuation(
        train_validation, text_column=winner_spec['text_column'], seed=SEED,
        repetitions=1, insertion_rate=0.08, augmented_weight=0.50,
    ) if winner_augmented else train_validation
)
final_model, final_diagnostics = fit_candidate(
    winner_spec['model'], final_training, max_features=MAX_FEATURES,
    text_column=winner_spec['text_column'], model_parameters={'C': winner_spec['C']},
    flash_pseudo_weight=winner_spec['flash_weight'],
)
final_model.thresholds = high_recall_thresholds.copy()
final_model.review_margin = float(routing_choice.iloc[0]['margen'])
final_model.metadata = {
    **hybrid_meta,
    'schema_version': '4.0', 'outputs': COARSE_ORDER,
    'fine_labels_trained': False, 'flags_trained_as_categories': False,
    'winner_id': WINNER_ID, 'winner_spec': winner_spec,
    'text_column': winner_spec['text_column'],
    'context_radius': 1 if winner_spec['text_column'] == 'context_text' else 0,
    'test_rows_excluded': len(test_df), 'human_ids_excluded': len(human_ids),
    'threshold_policy': f'max precision subject to validation recall >= {MINIMUM_RECALL}',
    'standalone_moderation_authorized': False,
}
MODEL_PATH = OUTPUT_MODEL / 'moderador_grueso_alta_sensibilidad.joblib'
save_coarse_model(final_model, MODEL_PATH)
(OUTPUT_MODEL / 'manifiesto.json').write_text(
    json.dumps(final_model.metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)
hard_negatives = mine_flash_hard_negatives(
    final_model, train_validation, text_column=winner_spec['text_column'],
    top_n=2_000, max_per_video=3,
)
HARD_PATH = ROOT / 'datos' / 'processed' / 'flash_seguros_dificiles_post_adjudicacion_iter2.csv'
save_hard_negative_manifest(HARD_PATH, hard_negatives, {
    'rows': len(hard_negatives), 'max_per_video': 3, 'test_excluded': True,
    'selection': 'iteration 2: highest predicted coarse damage after human adjudication',
    'source_human_adjudications': hybrid_meta['human_hard_rows'],
    'labels_changed': False, 'requires_independent_review': True,
})
display(hard_negatives.head(10))
print('Modelo:', MODEL_PATH)
print('Manifiesto de revisión:', HARD_PATH)
print('Tiempo reajuste final:', round(final_diagnostics['training_seconds'], 2), 's')

## 8. Transformer español: ruta justificada, no resultado simulado

BETO es un BERT preentrenado en español y constituye una extensión razonable para capturar semántica contextual (Cañete et al., 2020). No se ejecuta aquí porque el entorno verificado dispone de PyTorch solo para CPU, no tiene `transformers` instalado y afinar aproximadamente 70 mil chunks no sería una comparación responsable de tiempo/costo. Una fase GPU debe predecir exactamente las mismas cinco etiquetas de daño o SEGURO, usar BCE ponderada o focal, comenzar con 3 épocas, máximo 5, y parada temprana en PR-AUC macro de daño. Nunca debe entrenar las etiquetas finas ni los flags como clases.

## Referencias metodológicas (APA 7)

Cañete, J., Chaperon, G., Fuentes, R., Ho, J.-H., Kang, H., & Pérez, J. (2020). *Spanish pre-trained BERT model and evaluation data*. PML4DC at ICLR 2020. https://users.dcc.uchile.cl/~jperez/papers/pml4dc2020.pdf

Cawley, G. C., & Talbot, N. L. C. (2010). On over-fitting in model selection and subsequent selection bias in performance evaluation. *Journal of Machine Learning Research, 11*, 2079–2107. https://www.jmlr.org/papers/v11/cawley10a.html

Feng, S. Y., Gangal, V., Wei, J., Chandar, S., Vosoughi, S., Mitamura, T., & Hovy, E. (2021). A survey of data augmentation approaches for NLP. In *Findings of ACL-IJCNLP 2021* (pp. 968–988). Association for Computational Linguistics. https://doi.org/10.18653/v1/2021.findings-acl.84

Karimi, A., Rossi, L., & Prati, A. (2021). AEDA: An easier data augmentation technique for text classification. In *Findings of EMNLP 2021* (pp. 2748–2754). Association for Computational Linguistics. https://doi.org/10.18653/v1/2021.findings-emnlp.234

Lin, T.-Y., Goyal, P., Girshick, R., He, K., & Dollár, P. (2017). Focal loss for dense object detection. In *Proceedings of ICCV 2017* (pp. 2980–2988). https://openaccess.thecvf.com/content_ICCV_2017/html/Lin_Focal_Loss_for_ICCV_2017_paper.html

Saito, T., & Rehmsmeier, M. (2015). The precision-recall plot is more informative than the ROC plot when evaluating binary classifiers on imbalanced datasets. *PLOS ONE, 10*(3), e0118432. https://doi.org/10.1371/journal.pone.0118432

Song, X., Petrak, J., & Roberts, A. (2018). A deep neural network sentence level classification method with context information. In *Proceedings of EMNLP 2018* (pp. 900–904). Association for Computational Linguistics. https://doi.org/10.18653/v1/D18-1107